# Text Chunking

**Definition:** Chunking is splitting a large document into smaller pieces ("chunks") before storing it for retrieval. Retrieval-Augmented Generation (RAG) works by finding the *most relevant* pieces of a document and handing only those to Claude — but that only helps if a chunk is small enough to be about one topic. A chunk that's too big drags in irrelevant context; a chunk that's too small loses the surrounding meaning.

There are a few ways to decide where the splits go:

- **By character count** — cut every N characters, with a little overlap so a split doesn't destroy a sentence at the boundary. Simple, but can still cut mid-sentence.
- **By sentence** — group a fixed number of sentences per chunk. Keeps prose intact, but chunk sizes vary.
- **By document structure** — split on the document's own headings (e.g. every `## ` in Markdown). If the author already organized the document into topics, this gives you one chunk per topic for free — the cleanest split when the structure is available.

We'll use structure-based chunking as the running example for the rest of this RAG section, since `report.md` is already organized into headed sections.


In [1]:
def chunk_by_section(document_text):
    import re

    pattern = r"\n## "
    return re.split(pattern, document_text)


In [2]:
with open("./report.md", "r") as f:
    text = f.read()

chunks = chunk_by_section(text)

print(f"{len(chunks)} chunks\n")
for i, chunk in enumerate(chunks):
    preview = chunk[:80].replace("\n", " ")
    print(f"[{i}] {len(chunk):>5} chars | {preview}...")


15 chunks

[0]    70 chars | # **Annual Interdisciplinary Research Review: Cross-Domain Insights** ...
[1]  1960 chars | Executive Summary  This report synthesizes the key findings and ongoing research...
[2]   854 chars | Table of Contents  1.  Executive Summary 2.  Table of Contents 3.  Methodology 4...
[3]  1135 chars | Methodology  The insights compiled within this Annual Interdisciplinary Research...
[4]  1178 chars | Section 1: Medical Research - Understanding XDR-471 Syndrome  This year saw sign...
[5]  1243 chars | Section 2: Software Engineering - Project Phoenix Stability Enhancements  The So...
[6]  1170 chars | Section 3: Financial Analysis - Q3 Performance and Outlook  Quarterly financial ...
[7]  1184 chars | Section 4: Scientific Experimentation - Characterization of Material Composite X...
[8]  1223 chars | Section 5: Legal Developments - Navigating IP Precedents and Regulatory Shifts  ...
[9]  1142 chars | Section 6: Product Engineering - Finalizing Model Zircon-5 Spec

## Other strategies

For reference, here are the character-based and sentence-based strategies mentioned above. Both take a block of text and return a list of chunk strings, the same shape as `chunk_by_section` — swap them in when a document has no clear heading structure to split on.


In [3]:
def chunk_by_char(text, chunk_size=150, chunk_overlap=20):
    chunks = []
    start_idx = 0

    while start_idx < len(text):
        end_idx = min(start_idx + chunk_size, len(text))
        chunks.append(text[start_idx:end_idx])
        start_idx = end_idx - chunk_overlap if end_idx < len(text) else len(text)

    return chunks


def chunk_by_sentence(text, max_sentences_per_chunk=5, overlap_sentences=1):
    import re

    sentences = re.split(r"(?<=[.!?])\s+", text)
    chunks = []
    start_idx = 0

    while start_idx < len(sentences):
        end_idx = min(start_idx + max_sentences_per_chunk, len(sentences))
        chunks.append(" ".join(sentences[start_idx:end_idx]))
        start_idx = max(0, start_idx + max_sentences_per_chunk - overlap_sentences)

    return chunks


In [4]:
char_chunks = chunk_by_char(text)
sentence_chunks = chunk_by_sentence(text)

print(f"chunk_by_char:     {len(char_chunks)} chunks")
print(f"chunk_by_sentence: {len(sentence_chunks)} chunks\n")

print("chunk_by_char[0]:")
print(repr(char_chunks[0]))

print("\nchunk_by_sentence[0]:")
print(repr(sentence_chunks[0]))

chunk_by_char:     141 chunks
chunk_by_sentence: 33 chunks

chunk_by_char[0]:
'# **Annual Interdisciplinary Research Review: Cross-Domain Insights**\n\n## Executive Summary\n\nThis report synthesizes the key findings and ongoing rese'

chunk_by_sentence[0]:
"# **Annual Interdisciplinary Research Review: Cross-Domain Insights**\n\n## Executive Summary\n\nThis report synthesizes the key findings and ongoing research efforts across the organization's diverse operational and R&D departments for the past fiscal year. Our strength lies in the cross-pollination of ideas and methodologies, driving innovation and addressing complex challenges that transcend traditional disciplinary boundaries. This year's review highlights significant progress in ten critical areas. Advances in **Medical Research** focused on the rare XDR-471 syndrome, yielding new diagnostic insights. Concurrently, **Software Engineering** tackled persistent stability issues, implementing key fixes identified through error cod